# Deep-pool optimization sweep — cap stringency study

Test whether relaxing the linearity caps (`λ_eta`, `λ_slope`) closes the gap to example.ipynb's visual quality. Three runs in escalating order:
1. `λ=10` — 10× weaker than current
2. `λ=1` — 100× weaker
3. `λ=0` — matches example.ipynb exactly (no caps)

All three use the same apparatus (depth=3m, 24 freqs, 48 actuators, 200×200 grid, L-BFGS, full Snell, 1500 iters). Each run prints peak `|η|/d` and peak `|∇η|` in the validity report so we can see how far past the 0.10 cap the optimizer goes.

Each run ~5-15 min on T4. Total ~30 min.

In [ ]:
# Setup: clone repo, install as package, verify GPU
import os, sys, subprocess

REPO_DIR = "/content/water_v2"
REPO_URL = "https://github.com/alexhrubin/water_v2.git"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", "python-rewrite", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"{REPO_DIR} already exists; run the `git pull` cell below to update.")

os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "-e", REPO_DIR], check=True)

import jax
print(f"JAX backend: {jax.default_backend()}  devices: {jax.devices()}")
from wavetank import Tank, build_propagator   # noqa: F401
print("wavetank imports OK")

In [ ]:
# Pull latest after editing scripts locally and `git push`-ing
!cd /content/water_v2 && git pull

In [ ]:
# Sweep: λ=10 (mild relax)
!python -u notebooks/deep_pool_max_apparatus.py --lambda_eta 10 --lambda_slope 10

In [ ]:
# Sweep: λ=1 (significant relax)
!python -u notebooks/deep_pool_max_apparatus.py --lambda_eta 1 --lambda_slope 1

In [ ]:
# Sweep: λ=0 (caps off — matches example.ipynb)
!python -u notebooks/deep_pool_max_apparatus.py --lambda_eta 0 --lambda_slope 0

In [ ]:
# Compare the three runs side-by-side
from IPython.display import Image, display, Markdown
for tag in ['eta10_slope10', 'eta1_slope1', 'eta0_slope0']:
    display(Markdown(f'### λ_eta=λ_slope={tag.replace("eta", "").replace("_slope", ", ")}'))
    display(Image(f'data/deep_pool_max/{tag}/comparison.png'))